# Access the shared INCLUDE Drive folder

This Colab notebook authenticates to Google Drive, lists the contents of the shared folder, downloads a selected file in chunks, and displays text or CSV contents when appropriate. The folder may contain large archives: do not extract them wholesale, and do not commit ZIP files or videos to Git.

Shared folder: `https://drive.google.com/drive/folders/1t4i02oj5QIEyCqH1ilePXRkf8ACMxjIm?usp=share_link`

## 1. Authenticate and mount Google Drive

Run this in Google Colab. Sign in with an account that can access the shared folder. If it is only visible under “Shared with me,” add a shortcut to My Drive or use the Drive API listing cell below with the folder shared to your account.

In [ ]:
from google.colab import drive, auth
from googleapiclient.discovery import build
from pathlib import Path

drive.mount("/content/drive")
auth.authenticate_user()
drive_service = build("drive", "v3")

SHARED_FOLDER_ID = "1t4i02oj5QIEyCqH1ilePXRkf8ACMxjIm"
MY_DRIVE_ROOT = Path("/content/drive/MyDrive")
OUTPUT_ROOT = MY_DRIVE_ROOT / "isl_full_dataset_output"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# If the shared ZIP is added as a My Drive shortcut, set this to its mounted path.
ZIP_PATH = MY_DRIVE_ROOT / "PATH_TO_SHARED_INCLUDE_ARCHIVE.zip"
print("Output folder:", OUTPUT_ROOT)
print("ZIP exists:", ZIP_PATH.is_file())

## 2. List files in the shared folder

This queries the Drive API by folder ID and paginates results. It lists names and metadata only; no file contents are downloaded in this section.

In [ ]:
def list_drive_folder(folder_id):
    items = []
    page_token = None
    while True:
        response = drive_service.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields="nextPageToken, files(id,name,mimeType,size,modifiedTime)",
            pageSize=1000,
            pageToken=page_token,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
        ).execute()
        items.extend(response.get("files", []))
        page_token = response.get("nextPageToken")
        if not page_token:
            break
    return items

folder_items = list_drive_folder(SHARED_FOLDER_ID)
for item in folder_items:
    kind = "folder" if item["mimeType"] == "application/vnd.google-apps.folder" else "file"
    print(f"{kind:6} {item['name']}  id={item['id']}  size={item.get('size', 'n/a')}")

if not folder_items:
    print("No visible items. Confirm that the signed-in Google account has access to the shared folder.")

## 3. Download a file from Google Drive

Select a file by name or ID. Downloads are written in chunks to the mounted Drive output directory. For the full INCLUDE ZIP, this copies the archive only; it does not extract it. Confirm there is enough free space before starting a large download.

In [ ]:
from googleapiclient.http import MediaIoBaseDownload
import io

# Choose one visible file. Leave as None to avoid downloading anything automatically.
SELECTED_FILE_ID = None  # Example: "Drive file ID from the listing above"
SELECTED_FILE_NAME = None  # Optional exact name to choose instead of an ID


def download_drive_file(file_id, destination, chunk_size=8 * 1024 * 1024):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    request = drive_service.files().get_media(fileId=file_id, supportsAllDrives=True)
    with destination.open("wb") as output:
        downloader = MediaIoBaseDownload(output, request, chunksize=chunk_size)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if status:
                print(f"Download {status.progress() * 100:.1f}%")
    return destination


selected = next(
    (item for item in folder_items if item["id"] == SELECTED_FILE_ID), None
) if SELECTED_FILE_ID else next(
    (item for item in folder_items if item["name"] == SELECTED_FILE_NAME), None
) if SELECTED_FILE_NAME else None

if selected is None:
    print("Set SELECTED_FILE_ID or SELECTED_FILE_NAME after reviewing the file listing; no download was started.")
else:
    safe_name = Path(selected["name"]).name
    downloaded_path = download_drive_file(selected["id"], OUTPUT_ROOT / safe_name)
    print("Downloaded to:", downloaded_path)

In [ ]:
if "downloaded_path" in globals() and downloaded_path.suffix.lower() == ".zip":
    ZIP_PATH = downloaded_path
    print("ZIP_PATH set to downloaded archive:", ZIP_PATH)
else:
    print("ZIP_PATH remains configured at:", ZIP_PATH)

## 4. Read and display file contents

Text and CSV files are previewed with a size limit. ZIP archives are handled by listing their central directory only; this does not extract archive members.

In [ ]:
import csv
import zipfile

if "downloaded_path" not in globals():
    print("Select and download a file in Section 3 first.")
elif downloaded_path.suffix.lower() == ".zip":
    with zipfile.ZipFile(downloaded_path) as archive:
        video_counts = {}
        for info in archive.infolist():
            if info.is_dir():
                continue
            member = Path(info.filename)
            if member.suffix.lower() in {".avi", ".m4v", ".mkv", ".mov", ".mp4", ".mpeg", ".mpg", ".webm"}:
                video_counts[member.parent.name] = video_counts.get(member.parent.name, 0) + 1
    print("Archive video counts (listed only; nothing extracted):")
    for label, count in sorted(video_counts.items()):
        print(f"{label}: {count}")
elif downloaded_path.suffix.lower() in {".txt", ".csv", ".md", ".json"}:
    max_preview_bytes = 1_000_000
    if downloaded_path.stat().st_size > max_preview_bytes:
        print("File is too large for an inline preview:", downloaded_path.stat().st_size, "bytes")
    elif downloaded_path.suffix.lower() == ".csv":
        with downloaded_path.open(newline="", encoding="utf-8-sig") as file:
            for row_index, row in enumerate(csv.reader(file)):
                print(row)
                if row_index >= 19:
                    print("Preview limited to 20 rows.")
                    break
    else:
        print(downloaded_path.read_text(encoding="utf-8", errors="replace"))
else:
    print("Binary/video file downloaded; not decoded as text:", downloaded_path.name)

## Full INCLUDE extraction and training

The next cells use the repository’s streaming extractor. They copy the chosen archive to Drive using chunked download, inspect its ZIP central directory without expanding it, write one temporary video at a time, and resume from existing valid `.npz` files after a reconnect. Do not sync the ZIP or videos back into the Git repository.

In [ ]:
import subprocess
import sys

REPO_URL = "https://github.com/aadyaasoni/ISL-AI-Assistant.git"
REPO_DIR = Path("/content/ISL-AI-Assistant")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", "full-dataset", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "requirements-preprocessing.txt"), "-r", str(REPO_DIR / "requirements-training.txt")], check=True)
print("Repository:", REPO_DIR)
print("Mounted archive:", ZIP_PATH)
assert ZIP_PATH.is_file(), "Set ZIP_PATH to the downloaded shared INCLUDE ZIP before continuing."

In [ ]:
# Parameters are intentionally explicit so a resumed Colab run uses the same locations and split seed.
MIN_VALID_SAMPLES = 15
FLUSH_EVERY = 25
SPLIT_SEED = 42
FULL_DATASET_OUTPUT = OUTPUT_ROOT / "full_dataset"

subprocess.run(
    [sys.executable, str(REPO_DIR / "scripts/extract_from_zip.py"), str(ZIP_PATH), "--list-only"],
    cwd=REPO_DIR,
    check=True,
)

In [ ]:
subprocess.run(
    [
        sys.executable,
        str(REPO_DIR / "scripts/extract_from_zip.py"),
        str(ZIP_PATH),
        "--output-root", str(FULL_DATASET_OUTPUT),
        "--hand-model", str(REPO_DIR / "models/mediapipe/hand_landmarker.task"),
        "--pose-model", str(REPO_DIR / "models/mediapipe/pose_landmarker_lite.task"),
        "--min-samples", str(MIN_VALID_SAMPLES),
        "--flush-every", str(FLUSH_EVERY),
        "--seed", str(SPLIT_SEED),
    ],
    cwd=REPO_DIR,
    check=True,
)

In [ ]:
quality_report_path = FULL_DATASET_OUTPUT / "class_quality_report.json"
quality_report = json.loads(quality_report_path.read_text())
print("Included classes:", quality_report["included_classes"])
print("Excluded classes and reasons:")
for label, details in quality_report["excluded_classes"].items():
    print(f"  {label}: {details['valid_samples']} valid; {details['reason']}")
print("Signer-independent split verified:", quality_report["signer_independent_split_verified"])
print(quality_report["signer_split_note"])

import pandas as pd
training_metadata = pd.read_csv(FULL_DATASET_OUTPUT / "training_metadata.csv")
display(training_metadata.groupby(["label", "split"]).size().unstack(fill_value=0))

In [ ]:
FULL_TRAIN_OUTPUT = OUTPUT_ROOT / "training_outputs" / "full_dataset"
subprocess.run(
    [
        sys.executable,
        str(REPO_DIR / "scripts/train_bilstm.py"),
        "--metadata", str(FULL_DATASET_OUTPUT / "training_metadata.csv"),
        "--output-dir", str(FULL_TRAIN_OUTPUT),
    ],
    cwd=REPO_DIR,
    check=True,
)

metrics = json.loads((FULL_TRAIN_OUTPUT / "metrics.json").read_text())
print("Validation macro-F1:", metrics["best_validation_macro_f1"])
print("Test macro-F1:", metrics["test_macro_f1"])
print("Per-class results:")
for label, values in metrics["per_class"].items():
    print(label, values)

In [ ]:
print("Calibration checks:")
print(json.dumps(metrics["confidence_calibration"], indent=2))

FULL_CLIPS_OUTPUT = OUTPUT_ROOT / "avatar" / "clips" / "full_dataset"
subprocess.run(
    [
        sys.executable,
        str(REPO_DIR / "scripts/build_avatar_clips.py"),
        "--metadata", str(FULL_DATASET_OUTPUT / "training_metadata.csv"),
        "--label-map", str(FULL_TRAIN_OUTPUT / "label_mapping.json"),
        "--clips-dir", str(FULL_CLIPS_OUTPUT),
    ],
    cwd=REPO_DIR,
    check=True,
)
subprocess.run(
    [
        sys.executable,
        str(REPO_DIR / "scripts/build_label_aliases.py"),
        "--label-map", str(FULL_TRAIN_OUTPUT / "label_mapping.json"),
        "--output", str(FULL_TRAIN_OUTPUT / "synonyms.json"),
    ],
    cwd=REPO_DIR,
    check=True,
)
print("Bring this output folder back after verifying license/attribution requirements:", OUTPUT_ROOT)

In [ ]:
# Select the new model for this Colab session; the legacy model remains selectable by setting this back.
import os
os.environ["ISL_CHECKPOINT_DIR"] = str(FULL_TRAIN_OUTPUT)
os.environ["ISL_DATASET_METADATA"] = str(FULL_DATASET_OUTPUT / "training_metadata.csv")
os.environ["ISL_AVATAR_CLIPS_DIR"] = str(FULL_CLIPS_OUTPUT)
os.environ["ISL_SYNONYMS_PATH"] = str(FULL_TRAIN_OUTPUT / "synonyms.json")

selected_labels = json.loads((FULL_TRAIN_OUTPUT / "label_mapping.json").read_text())
print("Supported signs from selected mapping:")
for label, _ in sorted(selected_labels.items(), key=lambda item: item[1]):
    print(" ", label)
print("To use the legacy 11-class model later, set ISL_CHECKPOINT_DIR to the repo's training_outputs/bilstm_baseline folder and use its matching clip/map paths.")

Training keeps the existing BiLSTM implementation and uses the quality-gated metadata. The source metadata has no signer IDs; the quality report explicitly states that signer-independent separation cannot be verified. The trained output is written to a new folder so the existing 11-class checkpoint is preserved.